In [ ]:
import cv2
import numpy as np

In [ ]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("웹캠을 열수 없습니다.")
while True:
    (ret, frame) = cap.read()
    if not ret:
        print("프레임 오류")
        break

    flip_frame = cv2.flip(frame, 1) # 좌우로 뒤집기
    (heights, width, _) = flip_frame.shape
    (center_x, center_y) = (width // 2, heights // 2)
    roi = flip_frame[center_y - 150:center_y + 150, center_x - 150:center_x + 150]
    cv2.rectangle(flip_frame, (center_x - 150, center_y - 150),(center_x + 150, center_y + 150), (0, 0, 255), 2)

    cv2.imshow("Webcam", flip_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('c' or 'C'):
        gray_image = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray_image = np.flip(gray_image, axis=1)
        cv2.imwrite('gray_image.png', gray_image)
        gaussian_blur = cv2.GaussianBlur(gray_image, (5, 5), 3)
        (_, otsh_thresh) = cv2.threshold(gaussian_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        cv2.imshow("OTSH", otsh_thresh)
        kernel = np.ones((5, 5), np.uint8)
        erosion = cv2.erode(otsh_thresh, kernel, iterations=5)
        cv2.imshow("EROSION", erosion)
        cv2.imwrite("digit_binary_image.png", erosion)
        img = cv2.imread("digit_binary_image.png", cv2.IMREAD_UNCHANGED)
        (h, w) = img.shape[:2]
        crop_size = 300
        (cx, cy) = w // 2, h  // 2
        half = crop_size // 2
        (x1, x2) = cx - half, cx + half
        (y1, y2) = cy - half, cy + half
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)

        crop = img[y1:y2, x1:x2]
        cv2.imshow("CROP", crop)
        reversed_image = cv2.bitwise_not(crop)
        cv2.imshow("REVERSED_IMAGE", reversed_image)
        cv2.imwrite("IMAGE_FOR_TEST.png", reversed_image)


    if cv2.waitKey(30) == 27:
        break
cap.release()
cv2.destroyAllWindows()